# 03 · Слои спроса и конкурентов из 2ГИС

Кофейни уже собраны в `03_2gis_all_coffee.ipynb`. Здесь тем же способом (рубрики + деление области на квадраты) собираем другие слои 2ГИС, которые влияют на выбор места кофейни. Слои идут в порядке ценности, потому что лимит демо-ключа ограничен (1 000 запросов на весь срок ключа). Сначала новое, чего нет в данных команды, потом 2ГИС-версии того, что у команды есть только из OSM.

| слой | рубрики 2ГИС (ID) | в 2ГИС по Алматы | зачем |
|---|---|---|---|
| `offices` | Бизнес-центры (13796), Коворкинг (104110) | 329 + 74 | офисный спрос: утренний кофе, обед |
| `bakeries` | Пекарни (111594), Кафе-кондитерские (112658) | 374 + 296 | заменители кофейни — кофе с выпечкой |
| `fitness` | Фитнес-клубы (268), Тренажёрные залы (267) | 470 + 298 | поток активных горожан |
| `education` | Университеты (232), Академии (230), Институты (231), Колледжи (246) | 135 + 29 + 21 + 166 | студенты; у команды эти данные только из OSM |
| `malls` | Торговые центры (611), ТРЦ (19499) | 312 + 38 | у нас ТРЦ только из OSM |
| `vending` | Кофейные автоматы (112719) | 877 | дешёвая альтернатива кофейне |

Число заведений взято из каталога рубрик 2ГИС на 2026-11-21. Это вся территория 2ГИС «Алматы», вместе с пригородами.

**Бюджет.** `GLOBAL_BUDGET` — сколько запросов можно потратить на все слои вместе, с учётом уже потраченных. Если 2ГИС откажет из-за исчерпанного лимита ключа, сбор остановится. Последний слой тогда останется неполным, но всё собранное сохранится. Неполные места видно по флагу `dgis_<слой>_complete_1000m`.

**Выходы**
- `data/processed/2gis_layers/<слой>.csv` — объекты слоя внутри области (город + 1 км); `id` хранится текстом.
- `data/processed/2gis_layers/summary.json` — запросы и статусы квадратов по слоям.
- `data/processed/features/dgis_demand_features.csv` — для каждого кандидата: число объектов слоя в 500 и 1 000 м и флаг полноты.

In [1]:
import json
from pathlib import Path
import sys

import geopandas as gpd
import numpy as np
import pandas as pd
import requests
import shapely

PROJECT_ROOT = (Path("..") if Path.cwd().name == "notebooks" else Path(".")).resolve()
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
from dgis_api import (  # общий клиент 2ГИС: notebooks/dgis_api.py
    CollectionStopped, RubricSearch, collect_area, covered_by_squares, load_area, parse_item, read_api_key,
)

BOUNDARY_PATH = PROJECT_ROOT / "data" / "processed" / "almaty_boundary.gpkg"
CANDIDATES_PATH = PROJECT_ROOT / "data" / "processed" / "candidate_locations.gpkg"
OUT_DIR = PROJECT_ROOT / "data" / "processed" / "2gis_layers"
SUMMARY_JSON = OUT_DIR / "summary.json"
FEATURES_CSV = PROJECT_ROOT / "data" / "processed" / "features" / "dgis_demand_features.csv"

CRS_METRIC = "EPSG:32643"
N_CANDIDATES = 2822
REGION_ID = 67
AREA_BUFFER_M = 1000
MIN_SIDE_M = 250
RADII_M = (500, 1000)

LAYERS = {  # порядок = приоритет при ограниченном лимите
    "offices": {13796: "Бизнес-центры", 104110: "Коворкинг"},
    "bakeries": {111594: "Пекарни", 112658: "Кафе-кондитерские"},
    "fitness": {268: "Фитнес-клубы", 267: "Тренажёрные залы"},
    "education": {232: "Университеты", 230: "Академии", 231: "Институты", 246: "Колледжи"},
    "malls": {611: "Торговые центры", 19499: "Торгово-развлекательные центры"},
    "vending": {112719: "Кофейные автоматы"},
}


def layer_search(rubrics):
    return RubricSearch(rubric_ids=tuple(rubrics), region_id=REGION_ID, min_side_m=MIN_SIDE_M)


def layer_checkpoint(name, out_dir=OUT_DIR):
    return out_dir / f"{name}_progress.sqlite3"


city = gpd.read_file(BOUNDARY_PATH).to_crs(CRS_METRIC)
area = shapely.union_all(city.buffer(AREA_BUFFER_M).to_numpy())
print(f"Область: {area.area / 1e6:.0f} км²; слоёв: {len(LAYERS)}")

Область: 905 км²; слоёв: 6


## Сбор по слоям с общим бюджетом

`collect_layers` обходит слои по порядку и отдаёт каждому остаток общего бюджета. Если слой остановился (кончился бюджет, лимит ключа или ошибка 2ГИС), следующие слои не запускаются. При повторном запуске всё продолжается с того же места.

In [2]:
def attempts_of(checkpoint):
    if not Path(checkpoint).is_file():
        return 0
    return load_area(checkpoint)[2]["attempts"]


def collect_layers(layers, *, area, crs, key, get, global_budget, out_dir=OUT_DIR, interval=1.0, **clock):
    """Слои по порядку; каждому — остаток общего бюджета. Возвращает сводку по слоям."""
    report = {}
    for name, rubrics in layers.items():
        checkpoint = layer_checkpoint(name, out_dir)
        spent_total = sum(attempts_of(layer_checkpoint(n, out_dir)) for n in layers)
        own = attempts_of(checkpoint)
        left = global_budget - spent_total
        if left <= 0:
            report[name] = {"skipped": "общий бюджет исчерпан"}
            break
        result = collect_area(layer_search(rubrics), checkpoint, area=area, crs=crs, key=key, get=get,
                              max_requests_total=own + left, interval=interval, **clock)
        report[name] = result
        if result["stop_reason"]:
            break
    return report

## Офлайн-тесты

Работают на имитации 2ГИС, как в `03_2gis_all_coffee.ipynb`. Проверяется:
- общий бюджет делится между слоями по порядку;
- при остановке слоя следующие не запускаются;
- повторный запуск продолжает сбор без повторных запросов;
- подсчёт объектов в радиусе.

In [3]:
import tempfile
import unittest
from unittest.mock import Mock

from pyproj import Transformer


class FakeDGIS:
    """Имитация /3.0/items по рубрикам: у каждого места свой rubric_id."""

    def __init__(self, xy, rubric_ids, crs=CRS_METRIC):
        self.xy = np.asarray(xy, dtype=float)
        self.rubric_ids = np.asarray(rubric_ids)
        lon, lat = Transformer.from_crs(crs, "EPSG:4326", always_xy=True).transform(self.xy[:, 0], self.xy[:, 1])
        self.places = [{"id": str(70000001000000000 + i), "point": {"lon": lo, "lat": la},
                        "rubrics": [{"id": str(r), "name": f"r{r}", "kind": "primary"}]}
                       for i, (lo, la, r) in enumerate(zip(lon, lat, self.rubric_ids))]
        self.to_metric = Transformer.from_crs("EPSG:4326", crs, always_xy=True)
        self.calls = []

    def __call__(self, url, params, timeout, allow_redirects):
        self.calls.append(dict(params))
        cx, cy = self.to_metric.transform(*map(float, params["point"].split(",")))
        wanted = np.isin(self.rubric_ids, [int(r) for r in params["rubric_id"].split(",")])
        inside = np.flatnonzero(wanted & (np.hypot(self.xy[:, 0] - cx, self.xy[:, 1] - cy) <= params["radius"]))
        page, size = params["page"], params["page_size"]
        data = {"meta": {"code": 200},
                "result": {"items": [self.places[i] for i in inside[(page - 1) * size: page * size]], "total": len(inside)}}
        return Mock(status_code=200, headers={"Retry-After": None}, **{"json.return_value": data})


def count_within(candidates, points, radius_m):
    if len(points) == 0:
        return np.zeros(len(candidates), dtype=int)
    cand_pos, _ = points.sindex.query(candidates.geometry, predicate="dwithin", distance=radius_m)
    return np.bincount(cand_pos, minlength=len(candidates))


class LayerTests(unittest.TestCase):
    X0, Y0 = 650_000.0, 4_785_000.0

    def setUp(self):
        temp = tempfile.TemporaryDirectory()
        self.addCleanup(temp.cleanup)
        self.out = Path(temp.name)
        self.clock = [1_700_000_000.0]
        rng = np.random.default_rng(3)
        xy = rng.uniform([self.X0, self.Y0], [self.X0 + 3000, self.Y0 + 3000], size=(240, 2))
        self.fake = FakeDGIS(xy, rng.choice([1, 2, 3], size=240))
        self.area = shapely.box(self.X0, self.Y0, self.X0 + 3000, self.Y0 + 3000)
        self.layers = {"a": {1: "A"}, "b": {2: "B"}, "c": {3: "C"}}

    def run_layers(self, budget):
        return collect_layers(self.layers, area=self.area, crs=CRS_METRIC, key="mock-credential", get=self.fake,
                              global_budget=budget, out_dir=self.out, now=lambda: self.clock[0],
                              sleep=lambda s: self.clock.__setitem__(0, self.clock[0] + s))

    def test_budget_is_shared_in_order_and_resumes(self):
        full = self.run_layers(10_000)
        need = {name: full[name]["attempts"] for name in self.layers}
        for path in self.out.glob("*.sqlite3"):
            path.unlink()
        self.fake.calls.clear()
        partial = self.run_layers(need["a"] + 3)
        self.assertIsNone(partial["a"]["stop_reason"])
        self.assertIn("лимит", partial["b"]["stop_reason"])
        self.assertNotIn("c", partial)
        self.assertEqual(len(self.fake.calls), need["a"] + 3)
        resumed = self.run_layers(10_000)
        self.assertEqual(len(self.fake.calls), sum(need.values()))
        self.assertTrue(all(resumed[n]["cells"]["pending"] == 0 for n in self.layers))
        for name, rubric in (("a", "1"), ("b", "2"), ("c", "3")):
            items, _, _ = load_area(layer_checkpoint(name, self.out))
            expected = sum(1 for p, (x, y) in zip(self.fake.places, self.fake.xy)
                           if p["rubrics"][0]["id"] == rubric and self.area.covers(shapely.Point(x, y)))
            self.assertGreaterEqual(len({it["id"] for it in items if it["rubrics"][0]["id"] == rubric}), expected)

    def test_exhausted_budget_skips_rest(self):
        report = self.run_layers(0)
        self.assertEqual(report, {"a": {"skipped": "общий бюджет исчерпан"}})

    def test_count_within(self):
        cands = gpd.GeoDataFrame(geometry=gpd.points_from_xy([0, 1000], [0, 0]), crs=CRS_METRIC)
        pts = gpd.GeoDataFrame(geometry=gpd.points_from_xy([100, 400, 900, 5000], [0, 0, 0, 0]), crs=CRS_METRIC)
        self.assertEqual(count_within(cands, pts, 500).tolist(), [2, 1])   # 1000 м: только точка 900
        self.assertEqual(count_within(cands, pts.iloc[:0], 500).tolist(), [0, 0])


suite = unittest.defaultTestLoader.loadTestsFromTestCase(LayerTests)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
if not test_result.wasSuccessful():
    raise AssertionError("Проверки сбора слоёв завершились ошибкой.")

test_budget_is_shared_in_order_and_resumes (__main__.LayerTests.test_budget_is_shared_in_order_and_resumes) ... 

ok


test_count_within (__main__.LayerTests.test_count_within) ... 

ok


test_exhausted_budget_skips_rest (__main__.LayerTests.test_exhausted_budget_skips_rest) ... 

ok


----------------------------------------------------------------------
Ran 3 tests in 1.271s

OK


## Сбор

`GLOBAL_BUDGET` — запросы на все слои вместе. Число берётся из остатка лимита ключа: 1 000 минус потраченное на кофейни, догрузку и пробные запросы (их видно в личном кабинете 2ГИС). Если поставить больше остатка, сбор остановит сам 2ГИС, когда лимит кончится. Собранное при этом сохранится.

In [4]:
RUN_LIVE = False
GLOBAL_BUDGET = 600
MIN_INTERVAL_SECONDS = 1.0

if RUN_LIVE:
    with requests.Session() as session:
        layer_report = collect_layers(LAYERS, area=area, crs=CRS_METRIC, key=read_api_key(PROJECT_ROOT), get=session.get,
                                      global_budget=GLOBAL_BUDGET, interval=MIN_INTERVAL_SECONDS)
    for name, result in layer_report.items():
        print(name, result)
else:
    print("Запросы к 2ГИС выключены (RUN_LIVE = False).")

Запросы к 2ГИС выключены (RUN_LIVE = False).


## Результат по слоям

Для каждого слоя сохраняем объекты внутри области. Полнота: слой собран полностью, если у него нет квадратов `pending`. Если слой собран частично, у кандидатов, чей круг 1 000 м не целиком в квадратах `complete`, счётчики могут быть занижены. Их видно по флагу `dgis_<слой>_complete_1000m`.

In [5]:
layer_places, layer_cells, summary = {}, {}, {}
for name, rubrics in LAYERS.items():
    checkpoint = layer_checkpoint(name)
    if not checkpoint.is_file():
        summary[name] = {"status": "не собирался"}
        continue
    items, cells, info = load_area(checkpoint)
    places = pd.DataFrame([parse_item(it) for it in items]) if items else pd.DataFrame(columns=["id", "lon", "lat", "rubric_names"])
    if len(places):
        names = places["rubric_names"].fillna("").str.split(";").map(set)
        for rubric_id, rubric_name in rubrics.items():
            places[f"rubric_{rubric_id}"] = names.map(lambda r, n=rubric_name: n in r)
        geom = gpd.GeoSeries(gpd.points_from_xy(places["lon"], places["lat"]), crs="EPSG:4326").to_crs(CRS_METRIC)
        places["x"], places["y"] = geom.x.round(2).to_numpy(), geom.y.round(2).to_numpy()
        places = places[places["lon"].notna() & shapely.covers(area, geom.to_numpy())]
        places = places[~places["temporary_closed"]].sort_values("id").reset_index(drop=True)
    pending = sum(c["status"] == "pending" for c in cells)
    layer_places[name], layer_cells[name] = places, cells
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    places.to_csv(OUT_DIR / f"{name}.csv", index=False, lineterminator="\n")
    summary[name] = info | {"pending_cells": pending, "places_in_area": len(places), "complete": pending == 0}
    print(f"{name}: {len(places)} объектов, запросов {info['attempts']}, квадраты {info['cells']}"
          + ("" if pending == 0 else f" — НЕ ДОСОБРАН, осталось {pending} квадратов"))
total_attempts = sum(s.get("attempts", 0) for s in summary.values())
print("Всего запросов на слои:", total_attempts)
SUMMARY_JSON.parent.mkdir(parents=True, exist_ok=True)
_ = SUMMARY_JSON.write_text(json.dumps(summary, ensure_ascii=False, indent=2, sort_keys=True) + "\n", encoding="utf-8")

offices: 373 объектов, запросов 105, квадраты {'pending': 0, 'split': 12, 'complete': 37, 'capped': 0, 'short': 0}


bakeries: 512 объектов, запросов 149, квадраты {'pending': 0, 'split': 21, 'complete': 63, 'capped': 0, 'short': 0}


fitness: 443 объектов, запросов 129, квадраты {'pending': 0, 'split': 17, 'complete': 51, 'capped': 0, 'short': 0}


education: 266 объектов, запросов 80, квадраты {'pending': 0, 'split': 11, 'complete': 34, 'capped': 0, 'short': 0}
malls: 194 объектов, запросов 52, квадраты {'pending': 0, 'split': 5, 'complete': 15, 'capped': 0, 'short': 0}
vending: 485 объектов, запросов 113, квадраты {'pending': 32, 'split': 24, 'complete': 40, 'capped': 0, 'short': 0} — НЕ ДОСОБРАН, осталось 32 квадратов
Всего запросов на слои: 628


## Признаки для кандидатов

Для каждого собранного слоя записываем:
- `dgis_<слой>_500m` и `dgis_<слой>_1000m` — число объектов в радиусе;
- `dgis_<слой>_complete_1000m` — круг 1 000 м целиком внутри полностью собранных квадратов.

Для слоя `offices` есть ещё `dgis_coworking_1000m`: коворкинги отдельно от бизнес-центров.

In [6]:
features = None
if layer_places:
    grid = gpd.read_file(CANDIDATES_PATH)
    if "candidate_id" not in grid.columns:
        raise ValueError("В candidate_locations.gpkg нет столбца candidate_id. Пересоберите сетку через 02.")
    grid = grid[["candidate_id", "geometry"]].to_crs(CRS_METRIC)
    assert len(grid) == N_CANDIDATES and grid["candidate_id"].is_unique
    features = pd.DataFrame({"candidate_id": grid["candidate_id"].to_numpy(),
                             "x": grid.geometry.x.to_numpy(), "y": grid.geometry.y.to_numpy()})
    for name, places in layer_places.items():
        pts = gpd.GeoDataFrame(places, geometry=gpd.points_from_xy(places["x"], places["y"]), crs=CRS_METRIC)
        for radius in RADII_M:
            features[f"dgis_{name}_{radius}m"] = count_within(grid, pts, radius)
        if name == "offices" and "rubric_104110" in pts:
            features["dgis_coworking_1000m"] = count_within(grid, pts[pts["rubric_104110"]], 1000)
        features[f"dgis_{name}_complete_1000m"] = covered_by_squares(grid, layer_cells[name], 1000)
    assert features["candidate_id"].is_unique and len(features) == N_CANDIDATES
    FEATURES_CSV.parent.mkdir(parents=True, exist_ok=True)
    features.to_csv(FEATURES_CSV, index=False, lineterminator="\n")
    print(f"Сохранено: {FEATURES_CSV.relative_to(PROJECT_ROOT)}")
    display(features.drop(columns=["candidate_id", "x", "y"]).describe().T)
else:
    print("Слои ещё не собраны.")

Сохранено: data\processed\features\dgis_demand_features.csv


,count,mean,std,min,25%,50%,75%,max
dgis_offices_500m,2822.0,0.412119,1.443943,0.0,0.0,0.0,0.0,16.0
dgis_offices_1000m,2822.0,1.650248,4.812021,0.0,0.0,0.0,1.0,36.0
dgis_coworking_1000m,2822.0,0.303685,1.022391,0.0,0.0,0.0,0.0,10.0
dgis_bakeries_500m,2822.0,0.560241,1.597727,0.0,0.0,0.0,0.0,14.0
dgis_bakeries_1000m,2822.0,2.205174,5.213221,0.0,0.0,0.0,2.0,35.0
dgis_fitness_500m,2822.0,0.475195,1.342692,0.0,0.0,0.0,0.0,13.0
dgis_fitness_1000m,2822.0,1.930546,4.375271,0.0,0.0,0.0,2.0,32.0
dgis_education_500m,2822.0,0.308646,1.651503,0.0,0.0,0.0,0.0,28.0
dgis_education_1000m,2822.0,1.202339,4.884104,0.0,0.0,0.0,0.0,56.0
dgis_malls_500m,2822.0,0.208717,0.789883,0.0,0.0,0.0,0.0,10.0
